The following function computes $s_n[s_\mu]$ using the recursive formula:
## $$n h_n[s_\mu] = \sum_{k=1}^n p_k[s_\mu] h_{n-k}[s_\mu]$$

In [65]:
#the following is the monomial expansion of s_mu in d-variables
@cached_function
def mc_s_mu(mu,d):
    s = SymmetricFunctions(QQ).s()
    return list(s(mu).expand(d).monomial_coefficients().items())

# defaultdict is useful for adding values adict[key] += c means that if key is already a key in adict
# it will add c to the current value, if not, it makes c the initial value.
from collections import defaultdict

# this is a function that computes a dictionary for the terms in s_n[s_mu]
# do it for d variables + an extra variable for the z
# this has worked really well for computing s_n[s_m] for a fixed m and for enn=0,1,2,3,...
# because I want the output in x1,x2,...,xd plus an extra variable z the length
# of the tuples are going to be d+1
def is_weakly_decreasing(tup):
    """
    test if a tuple is weakly decreasing
    """
    return all(tup[i]>=tup[i+1] for i in range(len(tup)-1))
@cached_function
def S_d(d):
    """
    The symmetric group with a sign
    """
    return [(tuple(v-1 for v in p),p.sign()) for p in Permutations(d)]
def sn_smu(enn, mu, d):
    """
    compute the terms in s_n[s_mu] of length at most d by calling hn_smu
    (this is a wrapper function so that lists are accepted in a cached function)
    """
    return hn_smu(enn, Partition(mu), d)
@cached_function
def hn_smu(enn, mu, d):
    """
    compute h_n[s_mu] using the recursive formula
    n h_n[s_mu] = sum_{k=1}^n p_k[s_mu] h_{n-k}[s_mu]
    """
    if enn==0:
        return { (0,)*(d+1) : 1 }
    else:
        out = defaultdict(int)
        for k in range(1,enn+1):
            for (w,ccc) in mc_s_mu(mu,d):
                for (v,c) in hn_smu(enn-k,mu,d).items():
                    for (p,cc) in S_d(d):
                        wv = tuple(k*w[i]+v[p[i]]+i-p[i] for i in range(d))
                        if is_weakly_decreasing(wv):
                            out[wv]+=c*cc*ccc
        return dict({v+(0,):c//enn for (v,c) in out.items() if c!=0})

In [72]:
nvars = 2 # number of variables
mu = Partition([3])
BR = QQ[','.join('x'+str(i) for i in range(1,nvars+1))+',z'] # the polynomial ring QQ[x1,x2,..,xd,z]
BR.inject_variables()
et = sage.rings.polynomial.polydict.ETuple

Defining x1, x2, z


In this cell we put what we think is the denominator of ${\mathbb B}_{\mu}(x_1,x_2, \ldots, x_d; z)$

In [79]:
conj_den = (1-z**4*x1**6*x2**6)*(1-z*x1**2*x2**1)*(1-z*x1**3)

In [80]:
@cached_function
def conj_den_4():
    return BR(expand(
        conj_den
        )).dict().items()#
@cached_function
def den_coeff(d):
    return BR._from_dict({ et(list(t)[:-1]+[0]) : c for (t,c) in conj_den_4() if list(t)[-1]==d})
def calc_num(d):
    return sum(den_coeff(d-r)*BR(hn_smu(r,mu,nvars)) for r in range(d+1))
def diff_tup(p,q):
    return tuple([a-b for (a,b) in zip(q,p)])

In [81]:
out={}
for d in range(0,100):
    CC = calc_num(d)
    CCC = sorted(list(CC),key=lambda m: list(m[1].leading_item()[0])[::-1],reverse=True)
    #CCC=list(CC)
    if CC:
        out[d]=CCC[0][1].leading_item()[0]
        print(d,out[d], len(list(CC)),CCC[0])
        delta = 1
        if d-delta in out:
            print(diff_tup(out[d-delta],out[d]))
        print("*************")
    else:
        print(d, "####" )

0 (0, 0, 0) 1 (1, 1)
*************
1 (2, 1, 0) 1 (-1, x1^2*x2)
(2, 1, 0)
*************
2 (4, 2, 0) 1 (1, x1^4*x2^2)
(2, 1, 0)
*************
3 ####
4 ####
5 ####
6 ####
7 ####
8 ####
9 ####
10 ####
11 ####
12 ####
13 ####
14 ####
15 ####
16 ####
17 ####
18 ####
19 ####
20 ####
21 ####
22 ####
23 ####
24 ####
25 ####
26 ####
27 ####
28 ####
29 ####
30 ####
31 ####
32 ####
33 ####
34 ####
35 ####
36 ####
37 ####
38 ####
39 ####
40 ####
41 ####
42 ####
43 ####
44 ####
45 ####
46 ####
47 ####
48 ####
49 ####
50 ####
51 ####
52 ####
53 ####
54 ####
55 ####
56 ####
57 ####
58 ####
59 ####
60 ####
61 ####
62 ####
63 ####
64 ####
65 ####
66 ####
67 ####
68 ####
69 ####
70 ####
71 ####
72 ####
73 ####
74 ####
75 ####
76 ####
77 ####
78 ####
79 ####
80 ####
81 ####
82 ####
83 ####
84 ####
85 ####
86 ####
87 ####
88 ####
89 ####
90 ####
91 ####
92 ####
93 ####
94 ####
95 ####
96 ####
97 ####
98 ####
99 ####


Once you get a polynomial in the previous calculation, move on to giving the denominator:

In [85]:
out={}
nm_erator = 0
for d in range(0,20):
    CC = calc_num(d)
    nm_erator += z**d*CC
    print(z**d*CC)

1
-x1^2*x2*z
x1^4*x2^2*z^2
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0


In [86]:
Bmu_d = nm_erator/conj_den

In [87]:
Bmu_d

(x1^4*x2^2*z^2 - x1^2*x2*z + 1)/(-x1^11*x2^7*z^6 + x1^9*x2^6*z^5 + x1^8*x2^7*z^5 - x1^6*x2^6*z^4 + x1^5*x2*z^2 - x1^3*z - x1^2*x2*z + 1)